# JSON Utilities API Reference

Developer-facing statements defined in `libs/core/langchain_core/utils/json.py`.

# `parse_partial_json`

Parses JSON that may be incomplete, including input missing closing string, object, or array delimiters.

```python
parse_partial_json(
    s: str, # JSON string to parse
    *,
    strict: bool = False, # Value forwarded to json.loads
) -> Any # Parsed JSON value
```

The function first calls `json.loads` without modifying the input. When that fails, it escapes unescaped newline characters inside strings, closes an unterminated string, appends missing object and array delimiters, and progressively removes trailing characters until parsing succeeds.

Returns `None` when the input contains a mismatched closing `}` or `]`. If recovery fails after all characters have been removed, the original input is parsed again and its `json.JSONDecodeError` is propagated.

---

# `parse_json_markdown`

Parses JSON from plain text or a Markdown code fence.

```python
parse_json_markdown(
    json_string: str, # Plain JSON or Markdown containing JSON
    *,
    parser: Callable[[str], Any] = parse_partial_json, # Function used to parse the normalized JSON string
) -> Any # Parsed JSON value
```

`Callable` is imported only under `TYPE_CHECKING`; postponed annotations keep it available for type checking without importing it at runtime.

Before parsing, surrounding spaces, newlines, carriage returns, tabs, and backticks are stripped. Unescaped newlines, carriage returns, tabs, and quotes inside an `"action_input"` string are escaped.

If parsing the complete input raises `json.JSONDecodeError`, the function searches for content beginning with a triple-backtick fence and parses the captured content instead.

---

# `parse_and_check_json_markdown`

Parses a JSON object from Markdown and verifies that specified keys are present.

```python
parse_and_check_json_markdown(
    text: str, # Plain JSON or Markdown containing JSON
    expected_keys: list[str], # Keys that must exist in the parsed object
) -> dict[str, Any] # Parsed JSON object
```

The function checks only that the result is a dictionary and that every expected key exists; it does not validate the corresponding values.

Raises `OutputParserException` when parsing raises `json.JSONDecodeError`, when the parsed value is not a dictionary, or when an expected key is absent.

In [ ]:
# 1. Parse incomplete JSON
from langchain_core.utils.json import parse_partial_json # Import the partial JSON parser


incomplete_json = '{"name": "Saad", "skills": ["Python", "SQL"' # Create JSON missing the closing list and object brackets

parsed_data = parse_partial_json(incomplete_json) # Recover and parse the incomplete JSON

print(parsed_data) # Display the parsed Python dictionary
print(parsed_data["skills"]) # Display the recovered skills list

# It can also close an unfinished string:
unfinished_string = '{"message": "Welcome to LangChain' # Create JSON with an unfinished string and object
parsed_message = parse_partial_json(unfinished_string) # Complete and parse the unfinished JSON
print(parsed_message) # Display the recovered dictionary

# A mismatched closing bracket returns None:
malformed_json = '{"numbers": [1, 2, 3}' # Create JSON containing a mismatched closing character
result = parse_partial_json(malformed_json) # Try to parse the malformed JSON
print(result) # Display None because the closing characters do not match

In [ ]:
# 5. Handle a non-dictionary JSON value
from langchain_core.exceptions import OutputParserException # Import the output parsing exception
from langchain_core.utils.json import parse_and_check_json_markdown # Import the validation utility


list_response = '["Python", "SQL", "LangChain"]' # Create valid JSON that produces a list instead of a dictionary

try: # Begin exception handling
    result = parse_and_check_json_markdown( # Parse and validate the JSON value
        text=list_response, # Provide the JSON list
        expected_keys=["skills"], # Request a dictionary containing a skills key
    ) # Finish the validation call

except OutputParserException as error: # Catch the incorrect-result-type exception
    print("Parsing failed:") # Display an error heading
    print(error) # Display the error explaining that a dictionary was expected